# E-commerce Customer Journey & Conversion Analysis

## Sprint 2 — Data Understanding

### Objective

The objective of this notebook is to understand the structure, quality, and characteristics of the e-commerce behavioral dataset before performing data cleaning and analysis.

The dataset contains five months of customer interaction data, from October 2019 to February 2020, including product views, cart interactions, and purchases.

During this stage, we will:

- Inspect the available monthly datasets.
- Understand the dataset structure and level of granularity.
- Review columns and data types.
- Measure the number of records in each file.
- Identify missing values and potential data quality issues.
- Explore the types of customer events available.
- Determine an appropriate strategy for processing the full dataset efficiently.

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
raw_data_path = Path("../data/raw")

csv_files = sorted(raw_data_path.glob("*.csv"))

csv_files


[WindowsPath('../data/raw/2019-Dec.csv'),
 WindowsPath('../data/raw/2019-Nov.csv'),
 WindowsPath('../data/raw/2019-Oct.csv'),
 WindowsPath('../data/raw/2020-Feb.csv'),
 WindowsPath('../data/raw/2020-Jan.csv')]

In [4]:
sample = pd.read_csv(csv_files[0], nrows=5)
sample 

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-12-01 00:00:00 UTC,remove_from_cart,5712790,1487580005268456287,NaN,f.o.x,6.27,576802932,51d85cb0-897f-48d2-918b-ad63965c12dc
1,2019-12-01 00:00:00 UTC,view,5764655,1487580005411062629,NaN,cnd,29.05,412120092,8adff31e-2051-4894-9758-224bfa8aec18
2,2019-12-01 00:00:02 UTC,cart,4958,1487580009471148064,NaN,runail,1.19,494077766,c99a50e8-2fac-4c4d-89ec-41c05f114554
3,2019-12-01 00:00:05 UTC,view,5848413,1487580007675986893,NaN,freedecor,0.79,348405118,722ffea5-73c0-4924-8e8f-371ff8031af4
4,2019-12-01 00:00:07 UTC,view,5824148,1487580005511725929,NaN,NaN,5.56,576005683,28172809-7e4a-45ce-bab0-5efa90117cd5


## Dataset Granularity

Each row represents a specific user interaction with a product at a particular point in time.

Examples of events include:

- Viewing a product (`view`)
- Adding a product to the cart (`cart`)
- Removing a product from the cart (`remove_from_cart`)
- Purchasing a product (`purchase`)

Therefore, the dataset operates at the **user-product-event level**, meaning that the same user can appear in multiple rows as they interact with different products or perform multiple actions during their customer journey.

In [5]:
sample.info()


<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   event_time     5 non-null      str    
 1   event_type     5 non-null      str    
 2   product_id     5 non-null      int64  
 3   category_id    5 non-null      int64  
 4   category_code  0 non-null      float64
 5   brand          4 non-null      str    
 6   price          5 non-null      float64
 7   user_id        5 non-null      int64  
 8   user_session   5 non-null      str    
dtypes: float64(2), int64(3), str(4)
memory usage: 492.0 bytes


### Initial Data Structure Observations

The initial sample contains 9 columns and each row represents a user-product event at a specific point in time.

Initial observations:

- The `event_time` column is currently stored as a string and should be converted to `datetime` during the data cleaning stage.
- The `category_code` column is not represented with an appropriate data type in the initial sample. Since it represents product categories, a categorical data type may be more appropriate during data preparation.
- Missing values are present in `category_code` and `brand`.
- `product_id`, `category_id`, and `user_id` are stored as integer values, while `price` is stored as a numeric decimal value.
- Further analysis using a larger sample is required before drawing conclusions about missing values and data quality across the full dataset.

In [6]:
sample_large = pd.read_csv(csv_files[0], nrows=10000)

sample_large.shape

(10000, 9)

In [7]:
missing_values = sample_large.isnull().mean() * 100

missing_values.sort_values(ascending=False)

category_code    98.57
brand            44.13
user_session      0.09
event_time        0.00
event_type        0.00
category_id       0.00
product_id        0.00
price             0.00
user_id           0.00
dtype: float64

### Missing Values Analysis

The initial analysis reveals substantial missing data in `category_code` and `brand`.

- The `category_code` column is 98.57% incomplete, which may significantly limit its usefulness for analysis.
- The `brand` column contains 44.13% missing values, but still retains enough information to potentially support brand-level analysis.
- A small percentage of missing values (0.09%) was also identified in `user_session`.

These results are based on a sample of 10,000 records from the December dataset. Further analysis across the complete dataset is required before making final data-cleaning decisions.

In [8]:
sample_large["event_type"].value_counts()

event_type
view                4401
cart                2888
remove_from_cart    2034
purchase             677
Name: count, dtype: int64

The analysis of 10,000 events shows that product views are the most frequent interaction, with 4,401 events, followed by 2,888 cart events. A notable number of remove-from-cart events (2,034) were also observed, which may indicate potential friction during the cart stage. However, it is too early to draw conclusions because these figures represent event counts rather than unique users or confirmed funnel progression. Further analysis is required to determine whether the cart stage represents a significant drop-off point.

In [9]:
sample_large["user_id"].nunique()

1450

In [10]:
users_by_event = sample_large.groupby("event_type")["user_id"].nunique()

users_by_event.sort_values(ascending=False)

event_type
view                1350
cart                 358
remove_from_cart     221
purchase              59
Name: user_id, dtype: int64

The analysis shows that 59 unique users generated 677 purchase events. This may suggest repeat purchasing behavior; however, further analysis is required to determine whether these purchases represent multiple products within the same transaction or customers returning to make additional purchases over time.

In [11]:
sample_large["user_session"].nunique()

2410

In [12]:
sessions_per_user = sample_large.groupby("user_id")["user_session"].nunique()

sessions_per_user.describe()

count    1450.000000
mean        1.662069
std         8.330990
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max       289.000000
Name: user_session, dtype: float64

The analysis shows that most users have only one session, since the median and the 75th percentile are both equal to 1. However, at least one user has 289 sessions, which is significantly higher than the rest and may be influencing the average number of sessions per user. This unusual value should be investigated before drawing conclusions about customer behavior.

In [13]:
sessions_per_user.sort_values(ascending=False).head(10)

user_id
576802932    289
476577170    103
579179073     74
556730134     23
574974492     17
558093005     15
532955590     12
574710774     10
455300469     10
580001320     10
Name: user_session, dtype: int64

In [14]:
sample_large[sample_large["user_id"] == 576802932]["event_type"].value_counts()

event_type
remove_from_cart    343
view                287
purchase             64
cart                  5
Name: count, dtype: int64

### Session Activity Observation

Most users in the sample have only one recorded session, while a small number of users show substantially higher session activity.

The most active user recorded 289 unique sessions and 699 events in the sample. Their activity included 343 `remove_from_cart` events, 287 `view` events, 64 `purchase` events, and only 5 `cart` events.

This unusual event distribution indicates that raw event counts should not be interpreted as a sequential funnel without considering user sessions and event chronology. Further investigation will be required during the customer journey analysis.

In [15]:
sample_large.nunique().sort_values(ascending=False)

event_time       7009
product_id       4441
user_session     2410
user_id          1450
price             724
category_id       316
brand             150
category_code       7
event_type          4
dtype: int64

### Cardinality Analysis

The cardinality analysis provides additional insight into the diversity of the dataset.

- `product_id` contains 4,441 unique values, indicating a high level of product diversity within the 10,000-event sample.
- `brand` contains 150 unique values, showing that customer interactions involve a broad range of brands. However, this variable should be interpreted carefully because 44.13% of the `brand` values are missing in the sample.
- `event_type` contains only 4 unique values (`view`, `cart`, `remove_from_cart`, and `purchase`), confirming that it is a low-cardinality categorical variable representing the main types of customer interactions.
- `category_code` contains only 7 unique non-null values, but this result should not be interpreted as the total number of product categories because 98.57% of the values are missing in the sample.

Overall, the sample shows high diversity at the product level while customer behavior is represented through a small and clearly defined set of event types.

In [16]:
sample_large["price"].describe()

count    10000.000000
mean         8.451937
std         19.189618
min          0.000000
25%          1.730000
50%          3.970000
75%          6.350000
max        327.780000
Name: price, dtype: float64

### Price Distribution

The `price` variable contains no missing values within the 10,000-event sample.

The median price is $3.97, while the mean is higher at $8.45. Additionally, 75% of the observations have prices of $6.35 or less, while the maximum price reaches $327.78. This suggests a right-skewed price distribution in which a relatively small number of high-priced products may be increasing the mean.

The minimum recorded price is $0.00. Zero-price observations require further investigation during data preparation before determining whether they represent valid business cases or data quality issues.

High-priced observations should also be investigated rather than automatically treated as invalid outliers.

In [17]:
sample_large[sample_large["price"] == 0].shape[0]

10

In [18]:
sample_large[sample_large["price"] == 0]["event_type"].value_counts()

event_type
view    10
Name: count, dtype: int64

### Zero-Price Observation

The sample contains 10 records with a price of $0.00, representing only 0.1% of the 10,000 observations.

All zero-price records correspond to `view` events, and no zero-price `purchase` events were identified in the sample. Therefore, these observations do not directly affect purchase revenue within the current sample.

Further investigation across the complete dataset will be required before deciding whether zero-price records should be retained, corrected, or excluded during data preparation.

In [19]:
sample_large.duplicated().sum()

np.int64(557)

In [20]:
sample_large[sample_large.duplicated(keep=False)].sort_values(
    ["user_id", "event_time"]
).head(20)

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
5123,2019-12-01 03:44:19 UTC,remove_from_cart,5867083,1487580010779770966,NaN,de.lux,0.79,184386547,0f875f59-5cff-443a-850f-5494a4582d19
5124,2019-12-01 03:44:19 UTC,remove_from_cart,5867083,1487580010779770966,NaN,de.lux,0.79,184386547,0f875f59-5cff-443a-850f-5494a4582d19
5125,2019-12-01 03:44:21 UTC,remove_from_cart,5837916,1487580010779770966,NaN,NaN,0.63,184386547,0f875f59-5cff-443a-850f-5494a4582d19
5126,2019-12-01 03:44:21 UTC,remove_from_cart,5837916,1487580010779770966,NaN,NaN,0.63,184386547,0f875f59-5cff-443a-850f-5494a4582d19
5136,2019-12-01 03:45:08 UTC,remove_from_cart,5826809,1487580009445982239,NaN,NaN,0.63,184386547,0f875f59-5cff-443a-850f-5494a4582d19
5137,2019-12-01 03:45:08 UTC,remove_from_cart,5826809,1487580009445982239,NaN,NaN,0.63,184386547,0f875f59-5cff-443a-850f-5494a4582d19
5138,2019-12-01 03:45:09 UTC,remove_from_cart,5835870,1487580009445982239,NaN,NaN,0.79,184386547,0f875f59-5cff-443a-850f-5494a4582d19
5139,2019-12-01 03:45:09 UTC,remove_from_cart,5835870,1487580009445982239,NaN,NaN,0.79,184386547,0f875f59-5cff-443a-850f-5494a4582d19
9650,2019-12-01 05:52:19 UTC,remove_from_cart,5806552,1487580004916134735,NaN,NaN,10.63,201926223,71c92d99-3a13-401a-98b1-bc5c9979c028
9651,2019-12-01 05:52:19 UTC,remove_from_cart,5806552,1487580004916134735,NaN,NaN,10.63,201926223,71c92d99-3a13-401a-98b1-bc5c9979c028


In [21]:
duplicate_rows = sample_large[sample_large.duplicated()]

duplicate_rows["event_type"].value_counts()

event_type
remove_from_cart    486
cart                 71
Name: count, dtype: int64

### Duplicate Records Analysis

The 10,000-event sample contains 557 exact duplicate records, representing 5.57% of the sample.

Duplicate records are concentrated in two event types:

- `remove_from_cart`: 486 duplicated records
- `cart`: 71 duplicated records

No exact duplicates were identified among `view` or `purchase` events in the sample.

Approximately 87.3% of the duplicate records correspond to `remove_from_cart` events. This finding is particularly important because duplicate records may inflate raw event counts and distort the interpretation of customer journey behavior.

These records will be investigated and handled during the data preparation stage before calculating final funnel metrics.

In [22]:
pd.to_datetime(sample_large["event_time"]).agg(["min", "max"])

min   2019-12-01 00:00:00+00:00
max   2019-12-01 05:58:02+00:00
Name: event_time, dtype: datetime64[us, UTC]

### Temporal Coverage of the Sample

The 10,000-record sample covers approximately the first six hours of December 1, 2019, from 00:00:00 UTC to 05:58:02 UTC.

This confirms that the sample is useful for understanding the dataset structure and identifying potential data quality issues, but it is not representative of the entire December dataset or the complete five-month observation period.

Therefore, findings related to missing values, duplicate records, event distribution, user behavior, and price distribution should be treated as preliminary observations until the complete dataset is analyzed.

The timestamp values were successfully interpreted as UTC-aware datetime values, confirming that `event_time` can be converted from string to datetime during data preparation.

In [23]:
for file in csv_files:
    columns = pd.read_csv(file, nrows=0).columns.tolist()
    print(file.name, columns)

2019-Dec.csv ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']
2019-Nov.csv ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']
2019-Oct.csv ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']
2020-Feb.csv ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']
2020-Jan.csv ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']


### Schema Consistency Across Monthly Files

All five monthly CSV files contain the same 9 columns with consistent column names and ordering.

The common schema includes:

- `event_time`
- `event_type`
- `product_id`
- `category_id`
- `category_code`
- `brand`
- `price`
- `user_id`
- `user_session`

This structural consistency indicates that the monthly datasets can potentially be combined into a single dataset during the data preparation stage without requiring schema reconciliation.

The current file list is sorted alphabetically rather than chronologically, so temporal ordering should be handled explicitly when processing the monthly datasets.

In [24]:
row_counts = {}

for file in csv_files:
    total_rows = 0
    
    for chunk in pd.read_csv(file, chunksize=500000):
        total_rows += len(chunk)
    
    row_counts[file.name] = total_rows

row_counts

{'2019-Dec.csv': 3533286,
 '2019-Nov.csv': 4635837,
 '2019-Oct.csv': 4102283,
 '2020-Feb.csv': 4156682,
 '2020-Jan.csv': 4264752}

In [25]:
total_rows = sum(row_counts.values())

total_rows

20692840

### Dataset Size

The complete dataset contains 20,692,840 customer interaction events distributed across five monthly files:

- October 2019: 4,102,283 events
- November 2019: 4,635,837 events
- December 2019: 3,533,286 events
- January 2020: 4,264,752 events
- February 2020: 4,156,682 events

The dataset therefore contains more than 20.6 million behavioral records across a five-month observation period.

Due to the size of the dataset, loading all monthly files into memory simultaneously may be inefficient. Chunk-based processing will be considered during data preparation to perform data quality checks, transformations, and aggregation while controlling memory usage.

In [26]:
missing_counts = pd.Series(dtype="int64")

In [27]:
for file in csv_files:
    for chunk in pd.read_csv(file, chunksize=500000):
        missing_counts = missing_counts.add(
            chunk.isnull().sum(),
            fill_value=0
        )

In [28]:
missing_percentage = (missing_counts / total_rows) * 100

missing_percentage.sort_values(ascending=False)

category_code    98.291225
brand            42.319551
user_session      0.022220
event_time        0.000000
category_id       0.000000
event_type        0.000000
price             0.000000
product_id        0.000000
user_id           0.000000
dtype: float64

### Full-Dataset Missing Values Analysis

Missing values were evaluated across the complete dataset of 20,692,840 events using chunk-based processing.

The analysis identified three columns containing missing values:

- `category_code`: 98.29% missing
- `brand`: 42.32% missing
- `user_session`: 0.022% missing

The `category_code` variable is almost entirely incomplete, which may substantially limit its usefulness for product category analysis.

Although `brand` contains a significant proportion of missing values, approximately 57.68% of the records still contain brand information. Therefore, the variable may remain useful for brand-level analysis and should not be automatically discarded.

Missing values in `user_session` represent only a very small proportion of the dataset. However, because session identifiers are important for customer journey analysis, these records should be evaluated carefully during data preparation.

All remaining columns contain no missing values across the complete dataset.

In [29]:
event_counts = pd.Series(dtype="int64")

In [30]:
for file in csv_files:
    for chunk in pd.read_csv(file, chunksize=500000):
        event_counts = event_counts.add(
            chunk["event_type"].value_counts(),
            fill_value=0
        )

In [31]:
event_counts.sort_values(ascending=False)

event_type
view                9657821.0
cart                5768333.0
remove_from_cart    3979679.0
purchase            1287007.0
dtype: float64

### Full-Dataset Event Distribution

The complete dataset contains four customer interaction event types across 20,692,840 records:

- `view`: 9,657,821 events
- `cart`: 5,768,333 events
- `remove_from_cart`: 3,979,679 events
- `purchase`: 1,287,007 events

Product views are the most frequent interaction, while purchases are the least frequent event.

The dataset therefore contains the behavioral events required to analyze the customer journey from product discovery through cart interaction and purchase.

However, these values represent raw event counts rather than unique users or confirmed sequential funnel progression. A single user may generate multiple events, so these counts should not be interpreted directly as conversion rates.

User- and session-level analysis will be required to calculate meaningful funnel metrics.

In [32]:
global_min_date = None
global_max_date = None

In [33]:
for file in csv_files:
    for chunk in pd.read_csv(file, chunksize=500000, usecols=["event_time"]):
        
        event_times = pd.to_datetime(chunk["event_time"], utc=True)
        
        chunk_min = event_times.min()
        chunk_max = event_times.max()
        
        if global_min_date is None or chunk_min < global_min_date:
            global_min_date = chunk_min
            
        if global_max_date is None or chunk_max > global_max_date:
            global_max_date = chunk_max

In [34]:
print("First event:", global_min_date)
print("Last event:", global_max_date)

First event: 2019-10-01 00:00:00+00:00
Last event: 2020-02-29 23:59:59+00:00


### Full-Dataset Temporal Coverage

The complete dataset covers customer activity from October 1, 2019 at 00:00:00 UTC through February 29, 2020 at 23:59:59 UTC.

This confirms a continuous five-month observation window across the monthly datasets:

- October 2019
- November 2019
- December 2019
- January 2020
- February 2020

The timestamps are recorded in UTC, which should be preserved consistently during data preparation and time-based analysis.

This five-month observation period provides sufficient temporal coverage to analyze customer purchasing behavior, repeat purchases, and monthly customer cohorts.

In [35]:
zero_price_count = 0
negative_price_count = 0

In [36]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["price"]
    ):
        zero_price_count += (chunk["price"] == 0).sum()
        negative_price_count += (chunk["price"] < 0).sum()

In [37]:
print("Zero-price records:", zero_price_count)
print("Negative-price records:", negative_price_count)

Zero-price records: 104157
Negative-price records: 131


In [38]:
negative_event_counts = pd.Series(dtype="int64")

for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["price", "event_type"]
    ):
        negative_events = chunk.loc[
            chunk["price"] < 0,
            "event_type"
        ].value_counts()

        negative_event_counts = negative_event_counts.add(
            negative_events,
            fill_value=0
        )

negative_event_counts

event_type
purchase            126.0
remove_from_cart      3.0
view                  2.0
dtype: float64

### Price Data Quality Observations

The full dataset contains 104,157 zero-price records and 131 negative-price records.

Zero-price observations represent approximately 0.50% of the complete dataset, while negative-price observations represent only a very small proportion of the data.

However, 126 of the 131 negative-price records correspond to `purchase` events. This finding is important because negative purchase prices could directly affect revenue calculations.

The negative values should not be automatically removed, as they may represent returns, adjustments, data-entry issues, or another business process not explicitly documented in the dataset.

These records require further investigation during data preparation before calculating final revenue KPIs.

In [39]:
seen_hashes = set()
duplicate_count = 0

In [40]:
for file in csv_files:
    for chunk in pd.read_csv(file, chunksize=500000):

        row_hashes = pd.util.hash_pandas_object(
            chunk,
            index=False
        )

        for row_hash in row_hashes:
            if row_hash in seen_hashes:
                duplicate_count += 1
            else:
                seen_hashes.add(row_hash)

In [41]:
print("Exact duplicate records:", duplicate_count)

Exact duplicate records: 1109098


In [42]:
duplicate_percentage = (duplicate_count / total_rows) * 100

duplicate_percentage

5.35981527910137

### Full-Dataset Duplicate Analysis

Duplicate analysis was performed across the complete dataset of 20,692,840 records using row-level hashing to support memory-efficient processing.

A total of 1,109,098 records were identified as potential exact duplicates, representing approximately 5.36% of the complete dataset.

This result is consistent with the preliminary 10,000-record sample, where approximately 5.57% of the records were identified as duplicates.

The presence of more than one million potential duplicate records represents an important data quality issue because duplicated events could inflate customer interaction counts and distort funnel, conversion, and behavioral metrics.

These records should therefore be validated and handled during the data preparation stage before calculating final business KPIs.

Because duplicate detection was performed using 64-bit row hashes, the identified records are treated as potential exact duplicates until validated during data cleaning.

In [43]:
unique_users = set()

In [44]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["user_id"]
    ):
        unique_users.update(chunk["user_id"].unique())

In [45]:
len(unique_users)

1639358

### Unique Users

The complete dataset contains 1,639,358 unique users across 20,692,840 customer interaction events.

This confirms that individual users can generate multiple behavioral events throughout the five-month observation period.

The distinction between event-level and user-level activity is important for subsequent analyses. Raw event counts should not be interpreted as customer counts, and user-level metrics will be required for conversion, repeat-purchase, retention, and cohort analysis.

In [46]:
purchasing_users = set()

In [47]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["user_id", "event_type"]
    ):
        buyers = chunk.loc[
            chunk["event_type"] == "purchase",
            "user_id"
        ].unique()

        purchasing_users.update(buyers)

In [48]:
len(purchasing_users)

110518

### Purchasing Users

Across the complete five-month dataset, 110,518 unique users generated at least one `purchase` event.

This is substantially lower than the 1,287,007 recorded purchase events, confirming that individual purchasing users can generate multiple purchase events.

The dataset therefore distinguishes between purchase activity at the event level and purchasing behavior at the customer level.

Multiple purchase events from the same user should not automatically be interpreted as repeat purchases, since several purchase events may occur within the same session or transaction. Session and temporal analysis will be required to distinguish multi-product purchases from customers returning to purchase again.

In [49]:
purchasing_sessions = set()

In [50]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["event_type", "user_session"]
    ):
        purchase_sessions = chunk.loc[
            chunk["event_type"] == "purchase",
            "user_session"
        ].dropna().unique()

        purchasing_sessions.update(purchase_sessions)

In [51]:
len(purchasing_sessions)

155617

### Purchasing Sessions

The complete dataset contains 155,617 unique sessions with at least one `purchase` event.

The purchasing activity can therefore be summarized at three different levels:

- 1,287,007 purchase events
- 155,617 purchasing sessions
- 110,518 unique purchasing users

The substantially higher number of purchase events compared with purchasing sessions indicates that multiple purchase events can occur within the same session. Therefore, individual `purchase` events should not be interpreted as independent customer purchases.

Additionally, the number of purchasing sessions exceeds the number of purchasing users, suggesting that some customers may have purchased across multiple sessions. Further user-level and temporal analysis will be required to determine true repeat-purchase behavior.

In [52]:
purchase_sessions_by_user = {}

In [53]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["user_id", "user_session", "event_type"]
    ):
        purchases = chunk.loc[
            chunk["event_type"] == "purchase",
            ["user_id", "user_session"]
        ].dropna(subset=["user_session"]).drop_duplicates()

        for user_id, user_session in purchases.itertuples(index=False, name=None):
            purchase_sessions_by_user.setdefault(user_id, set()).add(user_session)

In [54]:
purchase_session_counts = pd.Series({
    user_id: len(sessions)
    for user_id, sessions in purchase_sessions_by_user.items()
})

purchase_session_counts.describe()

count    110518.000000
mean          1.408069
std           1.301675
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         122.000000
dtype: float64

In [55]:
repeat_buyers = (purchase_session_counts > 1).sum()

repeat_buyers

np.int64(23303)

In [56]:
multiple_session_rate = (repeat_buyers / len(purchasing_users)) * 100

multiple_session_rate

np.float64(21.08525308094609)

### Multiple Purchasing Sessions

Among the 110,518 unique purchasing users, 23,303 generated purchase events across more than one purchasing session.

This represents approximately 21.09% of all purchasing users.

The distribution of purchasing sessions is highly concentrated: at least 75% of purchasing users recorded only one purchasing session, while a smaller group generated multiple purchasing sessions. The maximum observed was 122 purchasing sessions for a single user.

These results suggest the presence of repeat purchasing behavior. However, multiple purchasing sessions should not yet be interpreted as confirmed repeat purchases. Temporal analysis is required to determine whether these sessions represent customers returning to purchase at different points in time.

In [57]:
purchase_dates_by_user = {}

In [58]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["user_id", "event_type", "event_time"]
    ):
        purchases = chunk.loc[
            chunk["event_type"] == "purchase",
            ["user_id", "event_time"]
        ].copy()

        purchases["purchase_date"] = pd.to_datetime(
            purchases["event_time"],
            utc=True
        ).dt.date

        purchases = purchases[
            ["user_id", "purchase_date"]
        ].drop_duplicates()

        for user_id, purchase_date in purchases.itertuples(
            index=False,
            name=None
        ):
            purchase_dates_by_user.setdefault(
                user_id,
                set()
            ).add(purchase_date)

In [59]:
purchase_day_counts = pd.Series({
    user_id: len(dates)
    for user_id, dates in purchase_dates_by_user.items()
})

purchase_day_counts.describe()

count    110518.000000
mean          1.396741
std           1.204383
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          83.000000
dtype: float64

In [60]:
repeat_purchase_users = (purchase_day_counts > 1).sum()

repeat_purchase_users

np.int64(22738)

In [61]:
repeat_purchase_rate = (
    repeat_purchase_users / len(purchasing_users)
) * 100

repeat_purchase_rate

np.float64(20.57402414086393)

### Repeat Purchase Behavior

Among the 110,518 unique purchasing users, 22,738 generated purchase events on more than one distinct calendar day during the five-month observation period.

Using purchasing activity on multiple calendar days as the operational definition of repeat purchasing, the preliminary Repeat Purchase Rate is approximately 20.57%.

This definition avoids treating multiple products purchased within the same session or multiple purchasing sessions occurring on the same day as separate repeat purchases.

The result indicates that approximately one in five purchasing users returned and purchased on another calendar day during the observation period.

This metric will be complemented by cohort and retention analysis to evaluate whether customers returned in subsequent months after their initial purchase.

In [62]:
unique_products = set()

In [63]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["product_id"]
    ):
        unique_products.update(chunk["product_id"].unique())

In [64]:
len(unique_products)

54571

In [65]:
purchased_products = set()

In [66]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["product_id", "event_type"]
    ):
        products = chunk.loc[
            chunk["event_type"] == "purchase",
            "product_id"
        ].unique()

        purchased_products.update(products)

In [67]:
len(purchased_products)

40777

In [68]:
purchased_product_rate = (
    len(purchased_products) / len(unique_products)
) * 100

purchased_product_rate


74.72283813747228

### Product Coverage and Purchasing Activity

The complete dataset contains 54,571 unique products, of which 40,777 registered at least one `purchase` event during the five-month observation period.

Approximately 74.72% of the observed products generated at least one purchase, while 25.28% did not record any purchase event.

This metric represents product purchase coverage rather than product conversion rate. A product is counted as purchased regardless of whether it generated one or thousands of purchase events.

Further product-level analysis will compare views, cart activity, purchases, and revenue to evaluate differences in product performance and conversion behavior.

In [69]:
unique_categories = set()

In [70]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["category_id"]
    ):
        unique_categories.update(chunk["category_id"].unique())

In [71]:
len(unique_categories)

525

In [72]:
category_mapping = {}

In [73]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["category_id", "category_code"]
    ):
        valid_categories = chunk.dropna(
            subset=["category_code"]
        )[["category_id", "category_code"]].drop_duplicates()

        for category_id, category_code in valid_categories.itertuples(
            index=False,
            name=None
        ):
            category_mapping.setdefault(
                category_id,
                set()
            ).add(category_code)

In [74]:
len(category_mapping)

18

In [75]:
category_code_counts = {
    category_id: len(codes)
    for category_id, codes in category_mapping.items()
}

category_code_counts

{1487580011970953351: 1,
 2193074740686488401: 1,
 1487580006350586771: 1,
 1487580012759482531: 1,
 1487580013053083824: 1,
 2193074740619379535: 1,
 2007399943458784057: 1,
 1487580010695884882: 1,
 1921723506584715388: 1,
 1487580008070251489: 1,
 2018287324474901238: 1,
 1487580012071616651: 1,
 1487580012147114126: 1,
 2022622168218599898: 1,
 1783999067181810204: 1,
 1487580008221246441: 1,
 1487580012549767321: 1,
 2235524499216793855: 1}

In [76]:
for category_id, codes in category_mapping.items():
    print(category_id, "→", list(codes)[0])

1487580011970953351 → furniture.bathroom.bath
2193074740686488401 → furniture.bathroom.bath
1487580006350586771 → appliances.environment.vacuum
1487580012759482531 → furniture.bathroom.bath
1487580013053083824 → stationery.cartrige
2193074740619379535 → furniture.living_room.cabinet
2007399943458784057 → apparel.glove
1487580010695884882 → accessories.bag
1921723506584715388 → accessories.cosmetic_bag
1487580008070251489 → appliances.personal.hair_cutter
2018287324474901238 → furniture.bathroom.bath
1487580012071616651 → apparel.glove
1487580012147114126 → furniture.bathroom.bath
2022622168218599898 → furniture.living_room.chair
1783999067181810204 → appliances.environment.air_conditioner
1487580008221246441 → appliances.environment.air_conditioner
1487580012549767321 → sport.diving
2235524499216793855 → appliances.personal.massager


### Category Information

The complete dataset contains 525 unique `category_id` values.

However, `category_code` has extremely limited coverage. Only 18 of the 525 unique category IDs have at least one known category code.

For the categories where a code is available, the relationship is consistent: each `category_id` is associated with only one `category_code`.

Therefore, `category_id` will be retained as the primary category identifier for category-level analysis. `category_code` may provide additional descriptive context where available, but its high missing-value rate (98.29%) makes it unsuitable as the primary variable for comprehensive category analysis.

Missing `category_code` values will not be inferred for categories without an observed mapping, since the available data does not provide sufficient evidence to assign them reliably.

In [77]:
unique_brands = set()

In [78]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["brand"]
    ):
        brands = chunk["brand"].dropna().unique()

        unique_brands.update(brands)

In [79]:
len(unique_brands)

273

In [80]:
brand_counts = pd.Series(dtype="int64")

In [81]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["brand"]
    ):
        chunk_brand_counts = chunk["brand"].value_counts()

        brand_counts = brand_counts.add(
            chunk_brand_counts,
            fill_value=0
        )

In [82]:
brand_counts.sort_values(ascending=False).head(10)

brand
runail       1528908.0
irisk        1033852.0
masura        861763.0
grattol       852591.0
bpw.style     434813.0
ingarden      430958.0
estel         360912.0
kapous        324394.0
jessnail      252996.0
uno           250377.0
dtype: float64

In [83]:
brand_purchase_counts = pd.Series(dtype="int64")

In [84]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["brand", "event_type"]
    ):
        purchases = chunk.loc[
            chunk["event_type"] == "purchase",
            "brand"
        ]

        chunk_purchase_counts = purchases.value_counts()

        brand_purchase_counts = brand_purchase_counts.add(
            chunk_purchase_counts,
            fill_value=0
        )

In [85]:
brand_purchase_counts.sort_values(ascending=False).head(10)

brand
runail       111408.0
irisk         73806.0
masura        49751.0
grattol       49542.0
bpw.style     37943.0
ingarden      27411.0
estel         19438.0
uno           17586.0
freedecor     17467.0
italwax       17105.0
dtype: float64

In [87]:
brand_view_counts = pd.Series(dtype="int64")

In [88]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["brand", "event_type"]
    ):
        views = chunk.loc[
            chunk["event_type"] == "view",
            "brand"
        ]

        chunk_view_counts = views.value_counts()

        brand_view_counts = brand_view_counts.add(
            chunk_view_counts,
            fill_value=0
        )

In [89]:
brand_view_counts.sort_values(ascending=False).head(10)

brand
runail       608857.0
grattol      426257.0
irisk        409737.0
masura       342247.0
estel        206045.0
kapous       196912.0
ingarden     175637.0
jessnail     172238.0
bpw.style    168656.0
concept      125202.0
dtype: float64

In [90]:
brand_performance = pd.DataFrame({
    "views": brand_view_counts,
    "purchases": brand_purchase_counts
}).fillna(0)

brand_performance.head()

,views,purchases
brand,,
airnails,27354.0,7504.0
almea,3556.0,146.0
andrea,407.0,6.0
ardell,39339.0,841.0
arganoil,221.0,5.0


In [91]:
brand_performance["view_to_purchase_pct"] = (
    brand_performance["purchases"]
    / brand_performance["views"]
) * 100

In [92]:
brand_performance["views"].describe()

count       273.000000
mean      20825.516484
std       61948.680857
min           1.000000
25%         647.000000
50%        2737.000000
75%       12640.000000
max      608857.000000
Name: views, dtype: float64

In [93]:
eligible_brands = brand_performance[
    brand_performance["views"] >= brand_performance["views"].median()
]

len(eligible_brands)

137

In [94]:
eligible_brands.sort_values(
    "view_to_purchase_pct",
    ascending=False
).head(10)

,views,purchases,view_to_purchase_pct
brand,,,
soleo,3320.0,1235.0,37.198795
severina,26333.0,9173.0,34.834618
swarovski,12909.0,4031.0,31.226276
benovy,7873.0,2180.0,27.689572
airnails,27354.0,7504.0,27.432917
farmstay,12303.0,3356.0,27.277900
dizao,8368.0,2201.0,26.302581
levissime,8807.0,2234.0,25.366186
skinlite,6402.0,1606.0,25.085911


### Brand Activity and Purchase Behavior

The dataset contains 273 unique brands with available brand information. However, approximately 42.32% of all records have missing brand values, which should be considered when interpreting brand-level results.

Brand activity is highly concentrated among a subset of brands. Runail recorded the highest overall interaction volume and also the highest number of purchase events.

To explore purchasing behavior relative to product visibility, an event-based view-to-purchase ratio was calculated. To reduce the influence of brands with very low activity, the comparison was limited to brands with at least the median number of view events (2,737), resulting in 137 eligible brands.

Among these brands, Soleo recorded the highest event-based view-to-purchase ratio at approximately 37.20%, with 3,320 view events and 1,235 purchase events. Severina recorded a slightly lower ratio of 34.83%, but at substantially higher volume, with 26,333 views and 9,173 purchases.

These results suggest that a higher volume of views does not necessarily translate into a higher view-to-purchase ratio.

This metric should not be interpreted as a true customer conversion rate because it compares aggregated view and purchase events rather than tracking the same users or sessions through the funnel.

In [95]:
unique_sessions = set()

In [96]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["user_session"]
    ):
        sessions = chunk["user_session"].dropna().unique()

        unique_sessions.update(sessions)

In [97]:
len(unique_sessions)

4535941

In [98]:
avg_sessions_per_user = (
    len(unique_sessions) / len(unique_users)
)

avg_sessions_per_user

2.7669008233711003

In [99]:
purchasing_session_rate = (
    len(purchasing_sessions) / len(unique_sessions)
) * 100

purchasing_session_rate

3.4307545005545705

### User Session Overview

The dataset contains 4,535,941 unique user sessions generated by 1,639,358 unique users during the five-month observation period.

On average, approximately 2.77 sessions were observed per user. This is an aggregate average and does not imply that every user generated the same number of sessions.

A total of 155,617 unique sessions contained at least one `purchase` event, representing approximately 3.43% of all observed sessions.

The 3.43% figure represents the proportion of observed sessions containing a purchase event and should not yet be interpreted as a formal session conversion rate. A session-level funnel analysis will later evaluate event progression and define the appropriate eligible population for conversion measurement.

In [100]:
global_min_price = float("inf")
global_max_price = float("-inf")

In [101]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["price"]
    ):
        chunk_min = chunk["price"].min()
        chunk_max = chunk["price"].max()

        global_min_price = min(global_min_price, chunk_min)
        global_max_price = max(global_max_price, chunk_max)

In [102]:
print("Minimum price:", global_min_price)
print("Maximum price:", global_max_price)

Minimum price: -79.37
Maximum price: 327.78


In [103]:
negative_price_rows = []

In [104]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000
    ):
        negative_rows = chunk[
            chunk["price"] < 0
        ]

        if not negative_rows.empty:
            negative_price_rows.append(negative_rows)

In [105]:
negative_prices_df = pd.concat(
    negative_price_rows,
    ignore_index=True
)

negative_prices_df.shape

(131, 9)

In [106]:
negative_prices_df["product_id"].value_counts().head(10)

product_id
5670257    39
5716857    37
5716859    31
5716855    12
5716861    12
Name: count, dtype: int64

In [107]:
negative_prices_df.groupby("product_id")["price"].agg(
    ["count", "min", "max"]
)

,count,min,max
product_id,,,
5670257,39,-15.87,-15.87
5716855,12,-7.94,-7.94
5716857,37,-23.81,-23.81
5716859,31,-47.62,-47.62
5716861,12,-79.37,-79.37


In [108]:
negative_product_ids = negative_prices_df["product_id"].unique()

negative_product_ids

array([5670257, 5716859, 5716857, 5716855, 5716861])

In [109]:
negative_products_all_rows = []

In [110]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000
    ):
        product_rows = chunk[
            chunk["product_id"].isin(negative_product_ids)
        ]

        if not product_rows.empty:
            negative_products_all_rows.append(product_rows)

In [111]:
negative_products_df = pd.concat(
    negative_products_all_rows,
    ignore_index=True
)

negative_products_df.shape

(131, 9)

In [112]:
pd.crosstab(
    negative_products_df["product_id"],
    negative_products_df["event_type"]
)

event_type,purchase,remove_from_cart,view
product_id,,,
5670257,39,0,0
5716855,12,0,0
5716857,34,2,1
5716859,30,1,0
5716861,11,0,1


Including purchase events with negative prices would reduce the calculated total revenue. Since the meaning of these negative values is unknown, they should be investigated before being included in revenue calculations.

In [113]:
negative_purchase_revenue = negative_prices_df.loc[
    negative_prices_df["event_type"] == "purchase",
    "price"
].sum()

negative_purchase_revenue

np.float64(-3825.419999999999)

In [114]:
total_purchase_revenue = 0

In [115]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["event_type", "price"]
    ):
        purchase_revenue = chunk.loc[
            chunk["event_type"] == "purchase",
            "price"
        ].sum()

        total_purchase_revenue += purchase_revenue

In [116]:
total_purchase_revenue

np.float64(6348004.870000001)

In [117]:
negative_revenue_impact_pct = (
    abs(negative_purchase_revenue) / total_purchase_revenue
) * 100

negative_revenue_impact_pct

np.float64(0.06026176851373459)

### Price Quality Assessment

The `price` variable ranges from -79.37 to 327.78 across the complete dataset.

A total of 131 records contain negative prices. These records are concentrated in only five product IDs, and each affected product consistently appears with a single negative price value. No positive or zero-price observations were found for these five products.

Of the 131 negative-price records, 126 correspond to `purchase` events, while the remaining five consist of three `remove_from_cart` events and two `view` events.

The negative-price purchase events sum to -3,825.42. Total purchase revenue calculated directly from the raw data is approximately 6,348,004.87, meaning that the magnitude of the negative-price purchases represents approximately 0.06% of total purchase revenue.

Although their aggregate financial impact is small, the business meaning of these negative prices cannot be determined from the available data. Therefore, they should be treated as a documented data-quality issue and investigated during Data Cleaning & Preparation rather than automatically converted, removed, or interpreted as refunds.

In [118]:
zero_price_event_counts = pd.Series(dtype="int64")

In [119]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["event_type", "price"]
    ):
        zero_price_events = chunk.loc[
            chunk["price"] == 0,
            "event_type"
        ].value_counts()

        zero_price_event_counts = zero_price_event_counts.add(
            zero_price_events,
            fill_value=0
        )

In [121]:
print(zero_price_event_counts.sort_values(ascending=False))

print(
    "Total zero-price records:",
    int(zero_price_event_counts.sum())
)

event_type
cart                49370.0
view                30490.0
remove_from_cart    24296.0
purchase                1.0
dtype: float64
Total zero-price records: 104157


In [122]:
zero_price_purchase_rows = []

In [123]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000
    ):
        zero_purchase = chunk[
            (chunk["event_type"] == "purchase") &
            (chunk["price"] == 0)
        ]

        if not zero_purchase.empty:
            zero_price_purchase_rows.append(zero_purchase)

In [124]:
zero_price_purchase_df = pd.concat(
    zero_price_purchase_rows,
    ignore_index=True
)

zero_price_purchase_df

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2020-01-13 10:24:06 UTC,purchase,5911801,1487580005671109489,NaN,NaN,0.0,562735465,3b8c48f2-b4bd-400e-a786-eace723dbffa


In [125]:
zero_purchase_product_id = zero_price_purchase_df.loc[0, "product_id"]

zero_purchase_product_id

np.int64(5911801)

In [126]:
zero_purchase_product_rows = []

In [127]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000
    ):
        product_rows = chunk[
            chunk["product_id"] == zero_purchase_product_id
        ]

        if not product_rows.empty:
            zero_purchase_product_rows.append(product_rows)

In [128]:
zero_purchase_product_df = pd.concat(
    zero_purchase_product_rows,
    ignore_index=True
)

zero_purchase_product_df.shape

(919, 9)

In [129]:
zero_purchase_product_df["price"].value_counts().sort_index()

price
0.00     50
1.71     43
1.73    826
Name: count, dtype: int64

In [130]:
pd.crosstab(
    zero_purchase_product_df["price"],
    zero_purchase_product_df["event_type"]
)

event_type,cart,purchase,remove_from_cart,view
price,,,,
0.00,19,1,7,23
1.71,18,4,4,17
1.73,280,66,207,273


The analysis shows that the zero-price purchase is unusual compared with the product's historical prices. It could potentially be associated with a promotion or an exceptional discount, but the available data does not provide enough evidence to confirm this explanation.

### Zero-Price Assessment

The dataset contains 104,157 records with a price of 0.00.

Most zero-price records correspond to non-purchase interactions:

- Cart: 49,370
- View: 30,490
- Remove from cart: 24,296
- Purchase: 1

Therefore, only one zero-price record corresponds to a purchase event, meaning that zero prices have a negligible direct impact on total purchase revenue.

The only zero-price purchase corresponds to product 5911801. This product appears 919 times in the dataset and is usually associated with positive prices. Of its 71 purchase events, 66 occurred at a price of 1.73, 4 at 1.71, and only 1 at 0.00.

This indicates that the zero-price purchase is unusual compared with the product's historical pricing behavior. It could potentially be associated with a promotion, exceptional discount, or another business condition; however, the available data does not provide sufficient evidence to confirm the underlying cause.

The zero-price purchase should therefore be documented as a data-quality anomaly and evaluated during Data Cleaning & Preparation rather than automatically removed or imputed.

In [131]:
for file in csv_files:
    columns = pd.read_csv(
        file,
        nrows=0
    ).columns.tolist()

    print(file.name, "->", columns)

2019-Dec.csv -> ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']
2019-Nov.csv -> ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']
2019-Oct.csv -> ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']
2020-Feb.csv -> ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']
2020-Jan.csv -> ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']


### Schema Consistency

All five monthly CSV files contain the same nine columns in the same order:

- `event_time`
- `event_type`
- `product_id`
- `category_id`
- `category_code`
- `brand`
- `price`
- `user_id`
- `user_session`

This confirms schema consistency across the October 2019 to February 2020 datasets.

Because the monthly files share the same structure, they can be integrated into a unified dataset during Data Cleaning & Preparation without requiring column-level schema reconciliation.

## Data Understanding Summary

The Data Understanding stage provided a comprehensive assessment of the five monthly e-commerce datasets covering the period from October 1, 2019 to February 29, 2020.

### Key Dataset Characteristics

- The dataset contains approximately 20.7 million customer interaction events across five monthly CSV files.
- All monthly files share the same nine-column schema.
- The data operates at the user-product-event level.
- The dataset contains four customer event types: `view`, `cart`, `remove_from_cart`, and `purchase`.
- 1,639,358 unique users were identified.
- 4,535,941 unique user sessions were identified.
- Users generated approximately 2.77 observed sessions on average.
- 110,518 unique users generated at least one purchase event.
- 155,617 unique sessions contained at least one purchase event.
- Approximately 3.43% of observed sessions contained at least one purchase event.

### Customer Purchasing Behavior

- Approximately 20.57% of purchasing users made purchases on more than one distinct day.
- Purchasing users generated an average of approximately 1.40 distinct purchase days.
- Approximately 21.09% of purchasing users generated purchases across more than one purchase session.

These findings indicate that repeat purchasing behavior is present in the dataset and can support subsequent retention and cohort analysis.

### Product and Brand Structure

- 54,571 unique products were observed.
- 40,777 products generated at least one purchase event.
- Approximately 74.72% of observed products generated at least one purchase event during the analysis period.
- 525 unique category IDs were identified.
- Only 18 category IDs had an available `category_code`, indicating substantial missing category classification information.
- 273 unique brands were identified among records with available brand information.
- Brand-level activity and purchasing behavior vary considerably across brands.

An exploratory event-based view-to-purchase ratio showed that higher interaction volume does not necessarily translate into a higher purchase-to-view ratio. This metric should not be interpreted as a true customer conversion rate because it is based on aggregated events rather than tracked user or session progression.

### Data Quality Findings

- Approximately 5.36% of records are exact duplicates.
- Missing values are concentrated primarily in `category_code` and `brand`.
- `user_session` contains only a very small proportion of missing values.
- 104,157 records have a price of 0.00, but only one corresponds to a purchase event.
- 131 records contain negative prices.
- Negative prices are concentrated in five product IDs.
- 126 of the 131 negative-price records correspond to purchase events.
- Negative-price purchases sum to approximately -3,825.42, representing only about 0.06% of total raw purchase revenue.
- The business meaning of negative prices cannot be determined from the available data.

### Data Preparation Requirements

The following issues should be addressed during Data Cleaning & Preparation:

1. Convert `event_time` from string to a timezone-aware datetime representation.
2. Integrate the five monthly files into a consistent analytical dataset.
3. Evaluate and remove exact duplicate records where appropriate.
4. Define a treatment strategy for missing `category_code` and `brand` values.
5. Evaluate the small number of missing `user_session` values.
6. Define a documented treatment strategy for zero and negative prices.
7. Preserve identifier columns as identifiers rather than treating them as quantitative measures.
8. Create analytical features required for funnel, purchasing behavior, and cohort analysis.
9. Optimize storage and processing because of the dataset's large size.

### Conclusion

The dataset provides sufficient behavioral, transactional, user, session, product, and temporal information to support the project's planned customer journey, purchasing behavior, and retention analyses.

The main limitations are missing product classification information, exact duplicate records, and a small number of anomalous price observations. These issues are identifiable and can be addressed systematically during the Data Cleaning & Preparation stage.